In [1]:
!pip install -q ultralytics transformers open-clip-torch hnswlib tqdm pandas accelerate

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.8 MB/s eta 0:00:00


In [2]:
# ============================================================
#  NOTEBOOK B — Condition B: Frozen CLIP + Frozen BLIP-2
#  Fused embedding: v = α·img_emb + (1-α)·txt_emb  (L2-normalised)
#
#  Seeds  : 039, 003, 113, 528  (team roll-numbers)
#  Alphas : 0.7, 0.5
#  Metrics: HR@K, Recall@K, mAP@K, NDCG@K  (K ∈ {5, 10, 15})
# ============================================================

# 

# ── Cell 1: Imports ──────────────────────────────────────────
import os, json, random, time
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm
import pandas as pd
import hnswlib
import open_clip
from ultralytics import YOLO
from torchvision import transforms
from transformers import Blip2Processor, Blip2ForConditionalGeneration, BlipProcessor, BlipForImageTextRetrieval

NOTEBOOK_START = time.time()

# ── Cell 2: Config ───────────────────────────────────────────
CFG = dict(
    img_root  = "/kaggle/input/datasets/fireworksbads/project2/project2/img/img",
    eval_file = "/kaggle/input/datasets/fireworksbads/project2/project2/eval/list_eval_partition.txt",
    yolo_weights = "/kaggle/input/datasets/fireworksbads/output/detect/train/weights/best.pt",
    clip_model    = "ViT-B-32",
    clip_pretrain = "openai",
    image_size    = 224,
    seeds  = [39, 3,528,113],
    alphas = [0.7],
    crop_dir   = "/kaggle/working/crops",
    index_dir  = "/kaggle/working/indexes_B",
    catalog    = "/kaggle/working/catalog.json",
    emb_cache  = "/kaggle/working/emb_cache",
    hnsw_M           = 32,
    hnsw_efConstruct = 200,
    hnsw_efSearch    = 50,
    yolo_batch     = 64,
    yolo_pad       = 10,
    blip_batch     = 64,
    clip_batch     = 256,
    max_new_tokens = 20,
    TIME_BUDGET_SECS = int(11.5 * 3600),
    top_k  = [5, 10, 15],
    device = "cuda" if torch.cuda.is_available() else "cpu",
)

os.makedirs(CFG["crop_dir"],  exist_ok=True)
os.makedirs(CFG["index_dir"], exist_ok=True)
os.makedirs(CFG["emb_cache"], exist_ok=True)

def _elapsed_h():
    return (time.time() - NOTEBOOK_START) / 3600
def _time_left():
    return CFG["TIME_BUDGET_SECS"] - (time.time() - NOTEBOOK_START)
def _check_budget(label=""):
    if _time_left() < 600:
        print(f"\n⚠  Time budget nearly exhausted at [{label}]. Saving & exiting.")
        raise SystemExit("Budget exceeded — partial results preserved.")

print(f"Device : {CFG['device']}")
print(f"Budget : {CFG['TIME_BUDGET_SECS'] / 3600:.1f} h")

# ── Cell 3: Read eval partition ───────────────────────────────
split_df = pd.read_csv(
    CFG["eval_file"], sep=r"\s+", header=0,
    names=["image_name", "item_id", "evaluation_status"], skiprows=1,
)
split_df = split_df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

gallery_df = split_df[split_df["evaluation_status"] == "gallery"].reset_index(drop=True)
query_df   = split_df[split_df["evaluation_status"] == "query"].reset_index(drop=True)

gallery_paths = [os.path.join(CFG["img_root"], p) for p in gallery_df["image_name"]]
gallery_items = gallery_df["item_id"].tolist()
gallery_item_counts = {}
for it in gallery_items:
    gallery_item_counts[it] = gallery_item_counts.get(it, 0) + 1

query_paths = [os.path.join(CFG["img_root"], p) for p in query_df["image_name"]]
query_items = query_df["item_id"].tolist()
print(f"Gallery: {len(gallery_df)}   Query: {len(query_df)}")

# ── Cell 4: Load YOLO ─────────────────────────────────────────
model_yolo = YOLO(CFG["yolo_weights"])
model_yolo.to(CFG["device"])
print("YOLO loaded.")

def _crop_path(img_path):
    fname = img_path.replace("/", "_").replace("\\", "_").lstrip("_") + ".jpg"
    return os.path.join(CFG["crop_dir"], fname)

def yolo_crop_batch(img_paths):
    """Batch YOLO crop with padding; largest-area box; disk cache."""
    pad = CFG["yolo_pad"]
    to_process = [p for p in img_paths if not os.path.exists(_crop_path(p))]
    if to_process:
        results = model_yolo.predict(to_process, device=CFG["device"], verbose=False)
        for img_path, result in zip(to_process, results):
            out   = _crop_path(img_path)
            img   = Image.open(img_path).convert("RGB")
            boxes = result.boxes
            if boxes is not None and len(boxes) > 0:
                areas = (boxes.xyxy[:, 2] - boxes.xyxy[:, 0]) * (boxes.xyxy[:, 3] - boxes.xyxy[:, 1])
                best = boxes[areas.argmax()]
                x1, y1, x2, y2 = map(int, best.xyxy[0].tolist())
                W, H = img.size
                img.crop((max(0, x1-pad), max(0, y1-pad),
                          min(W, x2+pad), min(H, y2+pad))).save(out)
            else:
                img.save(out)
    return [_crop_path(p) for p in img_paths]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Device : cuda
Budget : 11.5 h


/tmp/ipykernel_23/1913236905.py:76: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  split_df = split_df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


Gallery: 12612   Query: 14218
YOLO loaded.


In [3]:
# ── Cell 5: Load BLIP-2 (float16, frozen) ────────────────────
print(f"\n[{_elapsed_h():.2f}h] Loading BLIP-2 …")
blip_proc  = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
blip_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b", torch_dtype=torch.float16, device_map="auto",
).eval()
for p in blip_model.parameters():
    p.requires_grad = False
print("BLIP-2 loaded (frozen, float16, greedy decode).")

def caption_batch_fast(img_paths):
    captions = []
    BS = CFG["blip_batch"]
    for i in range(0, len(img_paths), BS):
        batch  = img_paths[i:i + BS]
        images = [Image.open(p).convert("RGB") for p in batch]
        inputs = blip_proc(images=images, return_tensors="pt",
                           padding=True).to(CFG["device"], torch.float16)
        with torch.no_grad():
            out = blip_model.generate(
                **inputs, max_new_tokens=CFG["max_new_tokens"],
                do_sample=False, num_beams=1,
            )
        decoded = blip_proc.batch_decode(out, skip_special_tokens=True)
        captions.extend([c.strip() for c in decoded])
    return captions


[0.00h] Loading BLIP-2 …


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/accelerate/utils/modeling.py:1598: UserWarning: The following device_map keys do not match any submodules in the model: ['query_tokens']
  warnings.warn(


BLIP-2 loaded (frozen, float16, greedy decode).


In [4]:

# ── Cell 6: Build shared catalog (crop + caption) — ONCE ─────
CATALOG = CFG["catalog"]
BS = CFG["yolo_batch"]

if os.path.exists(CATALOG):
    with open(CATALOG) as f:
        catalog = json.load(f)
    done      = {r["image_path"] for r in catalog}
    remaining = [p for p in gallery_paths if p not in done]
    print(f"[{_elapsed_h():.2f}h] Catalog loaded: {len(catalog)} done, {len(remaining)} remaining.")
else:
    catalog, remaining = [], gallery_paths

for i in tqdm(range(0, len(remaining), BS), desc="Catalog (crop+caption)"):
    _check_budget("catalog loop")
    batch = remaining[i:i + BS]
    try:
        crops    = yolo_crop_batch(batch)
        captions = caption_batch_fast(crops)
        for orig, crop, cap in zip(batch, crops, captions):
            item_id = orig.replace("\\", "/").split("/")[-2]
            catalog.append({"item_id": item_id, "image_path": orig,
                             "cropped_path": crop, "caption": cap})
    except Exception as e:
        print(f"  [skip batch {i}]: {e}")
    if (i // BS) % 50 == 0 and i > 0:
        with open(CATALOG, "w") as f:
            json.dump(catalog, f)

with open(CATALOG, "w") as f:
    json.dump(catalog, f, indent=2)
print(f"[{_elapsed_h():.2f}h] Catalog done — {len(catalog)} records")

Catalog (crop+caption): 100%|██████████| 198/198 [18:47<00:00,  5.70s/it]

[0.33h] Catalog done — 12612 records


In [5]:


# ── Cell 7: Offload BLIP-2 ───────────────────────────────────
del blip_model, blip_proc
torch.cuda.empty_cache()
print("BLIP-2 unloaded — VRAM freed.")

# ── Cell 8: Load CLIP ─────────────────────────────────────────
clip_model, _, _ = open_clip.create_model_and_transforms(
    CFG["clip_model"], pretrained=CFG["clip_pretrain"]
)
clip_model = clip_model.to(CFG["device"]).eval()
tokenizer  = open_clip.get_tokenizer(CFG["clip_model"])

eval_tf = transforms.Compose([
    transforms.Resize((CFG["image_size"], CFG["image_size"])),
    transforms.ToTensor(),
    transforms.Normalize((0.48145466, 0.4578275, 0.40821073),
                         (0.26862954, 0.26130258, 0.27577711)),
])
print("CLIP (pretrained, frozen) loaded.")

@torch.no_grad()
def encode_images_batched(img_paths):
    vecs = []
    BS   = CFG["clip_batch"]
    for i in tqdm(range(0, len(img_paths), BS), desc="  CLIP img encode", leave=False):
        batch = img_paths[i:i + BS]
        tensors = []
        for p in batch:
            try:    t = eval_tf(Image.open(p).convert("RGB"))
            except: t = torch.zeros(3, CFG["image_size"], CFG["image_size"])
            tensors.append(t)
        t_batch = torch.stack(tensors).to(CFG["device"])
        with torch.amp.autocast(CFG["device"], enabled=(CFG["device"] != "cpu")):
            feat = clip_model.encode_image(t_batch)
        vecs.append(F.normalize(feat, dim=-1).cpu().float().numpy())
    return np.concatenate(vecs, axis=0)

@torch.no_grad()
def encode_texts_batched(captions):
    vecs = []
    BS   = CFG["clip_batch"]
    for i in tqdm(range(0, len(captions), BS), desc="  CLIP txt encode", leave=False):
        batch = captions[i:i + BS]
        tok   = tokenizer(batch).to(CFG["device"])
        with torch.amp.autocast(CFG["device"], enabled=(CFG["device"] != "cpu")):
            feat = clip_model.encode_text(tok)
        vecs.append(F.normalize(feat, dim=-1).cpu().float().numpy())
    return np.concatenate(vecs, axis=0)


BLIP-2 unloaded — VRAM freed.


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


CLIP (pretrained, frozen) loaded.


In [6]:



# ── Cell 9: Cache CLIP embeddings ─────────────────────────────
gallery_crops    = [r["cropped_path"] for r in catalog]
gallery_captions = [r["caption"]      for r in catalog]
gallery_meta     = [{"item_id": r["item_id"], "image_path": r["image_path"],
                      "cropped_path": r["cropped_path"], "caption": r["caption"]}
                    for r in catalog]

img_cache = os.path.join(CFG["emb_cache"], "gallery_img_emb.npy")
txt_cache = os.path.join(CFG["emb_cache"], "gallery_txt_emb.npy")

if os.path.exists(img_cache):
    gal_img = np.load(img_cache)
    print(f"Loaded gallery img cache {gal_img.shape}")
else:
    print(f"[{_elapsed_h():.2f}h] Encoding gallery images …")
    gal_img = encode_images_batched(gallery_crops)
    np.save(img_cache, gal_img)

if os.path.exists(txt_cache):
    gal_txt = np.load(txt_cache)
    print(f"Loaded gallery txt cache {gal_txt.shape}")
else:
    print(f"[{_elapsed_h():.2f}h] Encoding gallery captions …")
    gal_txt = encode_texts_batched(gallery_captions)
    np.save(txt_cache, gal_txt)

# query image embeddings
q_img_cache = os.path.join(CFG["emb_cache"], "query_img_emb.npy")
print(f"\n[{_elapsed_h():.2f}h] Cropping & encoding query images …")

query_crops = []
for i in tqdm(range(0, len(query_paths), BS), desc="YOLO crop queries"):
    query_crops.extend(yolo_crop_batch(query_paths[i:i + BS]))

if os.path.exists(q_img_cache):
    q_img = np.load(q_img_cache)
    print(f"Loaded query img cache {q_img.shape}")
else:
    q_img = encode_images_batched(query_crops)
    np.save(q_img_cache, q_img)

# ── Cell 10: Metric helpers (TWO recall formulas) ──────────────
def hit_rate_at_k(ret, rel, k):
    """Hit Rate@K (binary): 1 if any relevant item in top-K, else 0."""
    return int(len(set(ret[:k]) & rel) > 0)

def recall_at_k(ret, rel, k, nr):
    """Recall@K (proportion): # relevant in top-K / total relevant in gallery."""
    hits = sum(1 for r in ret[:k] if r in rel)
    return hits / max(1, nr)

def ap_at_k(ret, rel, k, nr):
    h, s = 0, 0.0
    for rk, r in enumerate(ret[:k], 1):
        if r in rel: h += 1; s += h / rk
    return s / max(1, min(k, nr))

def ndcg_at_k(ret, rel, k, nr):
    dcg  = sum(1.0 / np.log2(i+2) for i, r in enumerate(ret[:k]) if r in rel)
    idcg = sum(1.0 / np.log2(i+2) for i in range(min(k, nr)))
    return dcg / idcg if idcg > 0 else 0.0

def compute_metrics(all_ret, all_rel, all_nr, top_k):
    out = {}
    for k in top_k:
        out[k] = {
            "HR":     (float(np.mean([hit_rate_at_k(r,rel,k) for r,rel in zip(all_ret,all_rel)])),
                       float(np.std ([hit_rate_at_k(r,rel,k) for r,rel in zip(all_ret,all_rel)]))),
            "Recall": (float(np.mean([recall_at_k(r,rel,k,nr) for r,rel,nr in zip(all_ret,all_rel,all_nr)])),
                       float(np.std ([recall_at_k(r,rel,k,nr) for r,rel,nr in zip(all_ret,all_rel,all_nr)]))),
            "mAP":    (float(np.mean([ap_at_k(r,rel,k,nr) for r,rel,nr in zip(all_ret,all_rel,all_nr)])),
                       float(np.std ([ap_at_k(r,rel,k,nr) for r,rel,nr in zip(all_ret,all_rel,all_nr)]))),
            "NDCG":   (float(np.mean([ndcg_at_k(r,rel,k,nr) for r,rel,nr in zip(all_ret,all_rel,all_nr)])),
                       float(np.std ([ndcg_at_k(r,rel,k,nr) for r,rel,nr in zip(all_ret,all_rel,all_nr)]))),
        }
    return out

[0.34h] Encoding gallery images …


[0.35h] Encoding gallery captions …



[0.35h] Cropping & encoding query images …


YOLO crop queries: 100%|██████████| 223/223 [04:33<00:00,  1.23s/it]


In [7]:

# ── Cell 11: Load BLIP ITM Re-ranker ──────────────────────────
print("\nLoading BLIP ITM model for semantic re-ranking...")
itm_proc = BlipProcessor.from_pretrained("Salesforce/blip-itm-base-coco")
itm_mdl  = BlipForImageTextRetrieval.from_pretrained(
    "Salesforce/blip-itm-base-coco", torch_dtype=torch.float16
).to(CFG["device"]).eval()

def itm_rerank(qcrop_path, cands, meta):
    """Re-ranks candidates using BLIP ITM scores from captions in meta."""
    valid_caps, valid_indices = [], []
    for i, c_idx in enumerate(cands):
        cap = meta[c_idx].get("caption", "")
        if cap:
            valid_caps.append(cap)
            valid_indices.append(i)
    if not valid_caps:
        return cands
    qcrop_img = Image.open(qcrop_path).convert("RGB")
    scores = []
    batch_size = 32
    for i in range(0, len(valid_caps), batch_size):
        v_caps = valid_caps[i:i + batch_size]
        i_batch = [qcrop_img] * len(v_caps)
        try:
            inp = itm_proc(images=i_batch, text=v_caps, return_tensors="pt", padding=True).to(CFG["device"])
            inp["pixel_values"] = inp["pixel_values"].half()
            with torch.no_grad():
                out = itm_mdl(**inp, use_itm_head=True)
                scores.extend(F.softmax(out.itm_score, dim=1)[:, 1].cpu().numpy().tolist())
        except:
            scores.extend([0.0] * len(v_caps))
    final_scores = np.zeros(len(cands))
    for score, idx in zip(scores, valid_indices):
        final_scores[idx] = score
    return [cands[i] for i in np.argsort(-final_scores, kind='stable')]

# ── Cell 12: Build HNSW helper (hnswlib, cosine) ──────────────
def build_hnsw(vectors):
    dim = vectors.shape[1]
    idx = hnswlib.Index(space='cosine', dim=dim)
    idx.init_index(max_elements=len(vectors),
                   ef_construction=CFG["hnsw_efConstruct"], M=CFG["hnsw_M"])
    idx.add_items(vectors, np.arange(len(vectors)))
    idx.set_ef(CFG["hnsw_efSearch"])
    return idx



Loading BLIP ITM model for semantic re-ranking...


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/456 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/895M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/472 [00:00<?, ?it/s]

BlipForImageTextRetrieval LOAD REPORT from: Salesforce/blip-itm-base-coco
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_encoder.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/895M [00:00<?, ?B/s]

In [8]:

# ── Cell 13: Main loop — 4 seeds × 2 alphas ───────────────────
all_results_B = {}
max_k = max(CFG["top_k"])

for seed in CFG["seeds"]:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    print(f"\n{'='*62}")
    print(f"  SEED {seed:03d}   [{_elapsed_h():.2f}h elapsed | {_time_left()/3600:.1f}h left]")
    print(f"{'='*62}")
    _check_budget(f"seed {seed}")

    perm       = np.random.permutation(len(query_paths))
    q_paths_s  = [query_paths[i]  for i in perm]
    q_crops_s  = [query_crops[i]  for i in perm]
    q_items_s  = [query_items[i]  for i in perm]
    q_embs_s   = q_img[perm]

    for alpha in CFG["alphas"]:
        _check_budget(f"seed {seed} alpha {alpha}")
        run_key = f"seed{seed:03d}_alpha{int(alpha*100):03d}"
        print(f"\n  ── {run_key} ──")

        # Fuse
        fused = alpha * gal_img + (1.0 - alpha) * gal_txt
        norms = np.linalg.norm(fused, axis=1, keepdims=True).clip(min=1e-8)
        fused_n = (fused / norms).astype("float32")

        index = build_hnsw(fused_n)
        tag   = f"B_{run_key}"
        index.save_index(os.path.join(CFG["index_dir"], f"index_{tag}.bin"))

        # Retrieve + ITM rerank
        all_ret, all_rel = [], []
        for qi in tqdm(range(len(q_paths_s)), desc="  Search & ITM Rerank", leave=False):
            qvec = q_embs_s[qi].reshape(1, -1).astype("float32")
            labels, distances = index.knn_query(qvec, k=max_k)
            cands = list(labels[0])
            reranked = itm_rerank(q_crops_s[qi], cands, gallery_meta)
            all_ret.append([gallery_meta[i]["item_id"] for i in reranked])
            all_rel.append({q_items_s[qi]})

        all_nr = [gallery_item_counts.get(list(rel)[0], 1) for rel in all_rel]
        metrics = compute_metrics(all_ret, all_rel, all_nr, CFG["top_k"])
        all_results_B[run_key] = metrics
        for k in CFG["top_k"]:
            hr,hrs = metrics[k]["HR"]; r,rs = metrics[k]["Recall"]
            m,ms   = metrics[k]["mAP"]; n,ns = metrics[k]["NDCG"]
            print(f"    K={k:2d}  HR={hr:.4f}±{hrs:.4f}  R={r:.4f}±{rs:.4f}  mAP={m:.4f}±{ms:.4f}  NDCG={n:.4f}±{ns:.4f}")

        del index, fused, fused_n

# ── Cell 14: Aggregate across seeds ───────────────────────────
print(f"\n\n{'='*80}")
print("  CONDITION B — Aggregated over 4 seeds")
print(f"{'='*80}")
print(f"{'Config':<28} {'K':>3}  {'HR@K':>12}  {'Recall@K':>12}  {'mAP':>12}  {'NDCG':>12}")
print("-" * 84)

aggregated_B = {}
for alpha in CFG["alphas"]:
    ak  = f"alpha{int(alpha*100):03d}"
    sms = [all_results_B[f"seed{s:03d}_{ak}"] for s in CFG["seeds"]
           if f"seed{s:03d}_{ak}" in all_results_B]
    if not sms: continue
    agg = {}
    for k in CFG["top_k"]:
        agg[k] = {
            met: (float(np.mean([sm[k][met][0] for sm in sms])),
                  float(np.std( [sm[k][met][0] for sm in sms])))
            for met in ["HR","Recall","mAP","NDCG"]
        }
    aggregated_B[ak] = agg
    for k in CFG["top_k"]:
        hr,hrs = agg[k]["HR"]; r,rs = agg[k]["Recall"]
        m,ms   = agg[k]["mAP"]; n,ns = agg[k]["NDCG"]
        print(f"  B α={alpha}           {k:>3}  {hr:.4f}±{hrs:.4f}  {r:.4f}±{rs:.4f}  {m:.4f}±{ms:.4f}  {n:.4f}±{ns:.4f}")
    print()

# ── Cell 15: Save results ──────────────────────────────────────
out_path = os.path.join(CFG["index_dir"], "results_B.json")
with open(out_path, "w") as f:
    json.dump({
        "seeds":      CFG["seeds"],
        "alphas":     CFG["alphas"],
        "per_seed":   {k: {str(kk): {m: list(v) for m, v in vv.items()}
                           for kk, vv in vs.items()}
                       for k, vs in all_results_B.items()},
        "aggregated": {k: {str(kk): {m: list(v) for m, v in vv.items()}
                           for kk, vv in vs.items()}
                       for k, vs in aggregated_B.items()},
    }, f, indent=2)

print(f"\n[{_elapsed_h():.2f}h] Results → {out_path}")
print("✅ Notebook B complete.")


  SEED 039   [0.45h elapsed | 11.1h left]

  ── seed039_alpha070 ──


    K= 5  HR=0.4106±0.4919  R=0.1527±0.2502  mAP=0.1219±0.2097  NDCG=0.1733±0.2523
    K=10  HR=0.5142±0.4998  R=0.2040±0.2797  mAP=0.1216±0.2036  NDCG=0.1869±0.2457
    K=15  HR=0.5656±0.4957  R=0.2334±0.2904  mAP=0.1237±0.2031  NDCG=0.1967±0.2451

  SEED 003   [1.53h elapsed | 10.0h left]

  ── seed003_alpha070 ──


    K= 5  HR=0.4110±0.4920  R=0.1528±0.2501  mAP=0.1220±0.2096  NDCG=0.1734±0.2522
    K=10  HR=0.5144±0.4998  R=0.2042±0.2799  mAP=0.1217±0.2035  NDCG=0.1870±0.2458
    K=15  HR=0.5657±0.4957  R=0.2335±0.2905  mAP=0.1238±0.2030  NDCG=0.1969±0.2451

  SEED 528   [2.60h elapsed | 8.9h left]

  ── seed528_alpha070 ──


    K= 5  HR=0.4107±0.4920  R=0.1526±0.2499  mAP=0.1219±0.2094  NDCG=0.1733±0.2521
    K=10  HR=0.5141±0.4998  R=0.2041±0.2798  mAP=0.1216±0.2034  NDCG=0.1869±0.2457
    K=15  HR=0.5654±0.4957  R=0.2334±0.2905  mAP=0.1237±0.2029  NDCG=0.1967±0.2450

  SEED 113   [3.68h elapsed | 7.8h left]

  ── seed113_alpha070 ──


    K= 5  HR=0.4105±0.4919  R=0.1528±0.2506  mAP=0.1221±0.2101  NDCG=0.1734±0.2526
    K=10  HR=0.5141±0.4998  R=0.2043±0.2803  mAP=0.1218±0.2041  NDCG=0.1870±0.2461
    K=15  HR=0.5654±0.4957  R=0.2336±0.2910  mAP=0.1239±0.2036  NDCG=0.1969±0.2455


  CONDITION B — Aggregated over 4 seeds
Config                         K          HR@K      Recall@K           mAP          NDCG
------------------------------------------------------------------------------------
  B α=0.7             5  0.4107±0.0002  0.1527±0.0001  0.1220±0.0001  0.1734±0.0001
  B α=0.7            10  0.5142±0.0001  0.2041±0.0001  0.1216±0.0001  0.1870±0.0001
  B α=0.7            15  0.5655±0.0001  0.2335±0.0001  0.1238±0.0001  0.1968±0.0001


[4.75h] Results → /kaggle/working/indexes_B/results_B.json
✅ Notebook B complete.
